# Attention & Transformers: Scaled Dot-Product, Multi-Head Attention & ViT

This interactive lab covers:
1. **Scaled Dot-Product Attention**: Computing attention weight matrices and causal future masks.
2. **Multi-Head Attention (MHA)**: Subspace projections and multi-head concatenation.
3. **Autoregressive Language Model (GPT-Style)**: Next-token prediction with causal masking.
4. **Vision Transformer (ViT) Patch Projection**: Transforming 2D spatial images into sequence token embeddings.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / 'code'))

import math
import torch
import torch.nn as nn
from transformer_engine import (
    ScaledDotProductAttention, MultiHeadAttention,
    SinusoidalPositionalEncoding, TransformerBlock,
    MiniTransformerLM, ViTPatchEmbedding
)

print('Transformer engine modules loaded successfully!')

## 1. Scaled Dot-Product Attention & Causal Masking

We compute self-attention over a toy 4-token sequence and apply an autoregressive causal mask to enforce that tokens only attend to past and current positions.

In [ ]:
torch.manual_seed(42)
seq_len = 4
d_k = 8

q = torch.randn(1, 1, seq_len, d_k)
k = torch.randn(1, 1, seq_len, d_k)
v = torch.randn(1, 1, seq_len, d_k)

# Create causal lower-triangular mask
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)

attn = ScaledDotProductAttention(dropout=0.0)
out, weights = attn(q, k, v, mask=causal_mask)

print('Causal Attention Weights (Rows = Query pos, Cols = Key pos):')
print(weights.squeeze().detach().numpy().round(3))
assert torch.allclose(weights.sum(dim=-1), torch.ones(1, 1, seq_len)), 'Attention weights must sum to 1!'

## 2. Multi-Head Attention Forward Pass

We verify multi-head projection where $d_{\text{model}} = 64$ is distributed across $h = 4$ attention heads of dimension $d_k = 16$.

In [ ]:
mha = MultiHeadAttention(d_model=64, num_heads=4, dropout=0.1)
x = torch.randn(2, 6, 64)  # Batch=2, SeqLen=6, Dim=64
out_mha = mha(x, x, x)

print(f'MHA Input Shape:  {x.shape}')
print(f'MHA Output Shape: {out_mha.shape}')
assert out_mha.shape == x.shape, 'MHA output shape must match input!'

## 3. Mini Autoregressive Transformer LM (GPT-Style)

We instantiate an autoregressive decoder-only Transformer language model and simulate a training optimization step.

In [ ]:
vocab_size = 200
model = MiniTransformerLM(
    vocab_size=vocab_size,
    d_model=64,
    num_heads=4,
    num_layers=2,
    max_len=32
)

total_params = sum(p.numel() for p in model.parameters())
print(f'Mini Transformer LM Total Parameters: {total_params:,}')

# Forward pass and next-token prediction loss
input_ids = torch.randint(0, vocab_size, (4, 16))  # Batch=4, SeqLen=16
target_ids = torch.randint(0, vocab_size, (4, 16))

logits = model(input_ids)
loss = nn.CrossEntropyLoss()(logits.view(-1, vocab_size), target_ids.view(-1))
loss.backward()

print(f'Forward Logits Shape: {logits.shape}')
print(f'Next-Token Cross-Entropy Loss: {loss.item():.4f}')

## 4. Vision Transformer (ViT) Patch Embedding

We convert a $32 \times 32$ image with $4 \times 4$ patches into a 1D sequence of $(32/4)^2 = 64$ patch tokens for vision transformer processing.

In [ ]:
vit_patcher = ViTPatchEmbedding(img_size=32, patch_size=4, in_channels=3, d_model=64)
img = torch.randn(2, 3, 32, 32)
tokens = vit_patcher(img)

print(f'Input Image Tensor Shape:      {img.shape}')
print(f'ViT Projected Patch Tokens:    {tokens.shape} (Batch, NumPatches, Dim)')
assert tokens.shape == (2, 64, 64)